# Fragrance Recommender met ChromaDB

RAG-recommender bovenop **ChromaDB** - dezelfde vector store als `fragrantica_chatbot.ipynb`, zodat beide notebooks consistent zijn. ChromaDB doet cosine-similarity, geeft per hit de afstand terug (zodat we similarity écht in de ranking kunnen meewegen) en ondersteunt een `where`-filter, zodat gender/rating in de database gefilterd worden in plaats van achteraf.

## Beste aanpak voor queries zoals 'spicy perfume for a date'

Het resultaat staat of valt met wat we embedden. Een goede `embedding_text` is een tekst-blob per parfum met zowel expliciete features (notes, brand, gender) als impliciete vibes uit de Fragrantica `description` en de user reviews:

1. **Een rijke tekst per parfum** - title + brand + gender + notes + description + reviews. De reviews bevatten echte gebruikerstaal (*evening*, *seductive*, *lasts all day*) die queries als "for a date" matcht zonder expliciete labels.
2. **Eén embedding per parfum** - `all-MiniLM-L6-v2` via sentence-transformers (zelfde model als de chatbot). Geen chunking: één parfum = één vector = één rij in de resul taten.
3. **Metadata meeopslaan** - rating, gender, brand, url - voor `where`-filtering en reranken.
4. **Re-ranken op rating** - bij gelijke similarity wint het hoger gerate parfum. Dat is wat een mens ook zou doen.
5. **Query expansion (optioneel)** - korte queries als "spicy for date" verrijken we met gangbare notes/vibe-woorden ("cinnamon, pepper, evening, seductive") voor betere recall.

## 0. Installatie

In [1]:
%pip install -q chromadb sentence-transformers pandas rank-bm25 openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import ast
import chromadb
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_colwidth', 120)
print('Imports OK')

C:\Users\alr3m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Data inladen en samenvoegen

We voegen drie bronnen samen tot één dataset:

1. **Fragrantica master** (`perfumes_table.csv`, 84k) - de originele dataset
2. **Filled-missing batches** (`fragnantica_missing_filled` + `missing_part1/2/3_filled`) - rijen die eerst leeg waren en die we aangevuld hebben via Parfumo. We matchen op `url` en overschrijven NaN-cellen in de master.
3. **Luckyscent** (`luckyscent_fragrances.csv`, 3k) - andere bron, ander schema. Heeft schone `gender` en `style` kolommen. We normaliseren naar gedeelde kolommen en concatten.

Daarna een kwaliteitsfilter: we houden alleen rijen met `reviews` en `notes`, want zonder notes valt er niks zinnigs te embedden. Reviews bevatten echte gebruikerstaal ("perfect for a date", "lasts all day") - dat is precies de vocabulaire die queries als *"spicy for a date"* matcht. Luckyscent valt hier af omdat die bron geen reviews heeft.

In [ ]:
# Fragrantica master + alle filled-missing batches 
master = pd.read_csv('fragrantica_data/perfumes_table.csv')

FILLED_FILES = [
    'fragrantica_data/fragnantica_missing_filled.csv',
    'fragrantica_data/missing_part1_filled.csv',
    'fragrantica_data/missing_part2_filled.csv',
    'fragrantica_data/missing_part3_filled.csv',
]
filled_parts = []
for fp in FILLED_FILES:
    part = pd.read_csv(fp)
    print(f'  {fp}: {len(part):,} rijen')
    filled_parts.append(part)
filled = pd.concat(filled_parts, ignore_index=True)
print(f'Filled-missing totaal (vóór dedupe): {len(filled):,}')

# Dedupe op url: behoud meest "ingevulde" rij per url (minste NaNs wint)
filled['_nonnull'] = filled.notna().sum(axis=1)
filled = (
    filled.sort_values('_nonnull', ascending=False)
          .drop_duplicates(subset='url', keep='first')
          .drop(columns='_nonnull')
)
print(f'Filled-missing na dedupe: {len(filled):,} unieke urls')

# Parfumo rating is 0-10, fragrantica is 0-5. normalized_rating is parfumo/10.
# Voor de filled rijen vervangen we 'rating' door normalized_rating × 5,
# zodat alles op 0-5 schaal staat.
if 'normalized_rating' in filled.columns:
    nr = pd.to_numeric(filled['normalized_rating'], errors='coerce')
    filled['rating'] = (nr * 5).where(nr.notna(), pd.to_numeric(filled['rating'], errors='coerce'))

# Drop kolommen die we niet meer nodig hebben vóór de merge
DROP_COLS = ['best_rating', 'rating_count', 'normalized_rating', 'parfumo_url']
filled = filled.drop(columns=[c for c in DROP_COLS if c in filled.columns])

# Merge in master (match op url) - update() raakt alleen bestaande kolommen
master = master.drop_duplicates(subset='url', keep='first').set_index('url')
filled_idx = filled.set_index('url')
master.update(filled_idx)
master = master.reset_index()

# --- Dedupe op (title, designer) - vangt Fragrantica's dubbele listings
# (zelfde parfum, twee URL-IDs zoals Gingham Love 77508 & 72692). Behoud de rij
# met de meeste non-null cellen (meestal degene mét reviews).
def _norm(s):
    return ' '.join(str(s).lower().split()) if pd.notna(s) else ''

master['_t'] = master['title'].apply(_norm)
master['_d'] = master['designer'].apply(_norm)
master['_nonnull'] = master.notna().sum(axis=1)

before_dedupe = len(master)
master = (
    master[master['_t'] != '']  # rijen zonder title kan je niet matchen
    .sort_values('_nonnull', ascending=False)
    .drop_duplicates(subset=['_t', '_d'], keep='first')
    .drop(columns=['_t', '_d', '_nonnull'])
    .reset_index(drop=True)
)
print(f'(Title, designer)-dedupe: {before_dedupe:,} → {len(master):,}  (-{before_dedupe-len(master)})')
print(f'\nFragrantica (incl. filled, gededupliceerd): {len(master):,} parfums')

# Luckyscent normaliseren naar gedeelde schema
ls = pd.read_csv('luckyscent_data/luckyscent_fragrances.csv')
ls_norm = pd.DataFrame({
    'url':         ls['url'],
    'title':       ls['name'].fillna('') + ' - ' + ls['brand'].fillna(''),
    'designer':    ls['brand'],
    'notes':       ls['notes'],
    'description': ls['description'],
    'rating':      pd.NA,
    'reviews':     pd.NA,
    'gender_raw':  ls['gender'],
    'style':       ls['style'],
})
print(f'Luckyscent: {len(ls_norm):,} parfums')

# Concat 
df = pd.concat([master, ls_norm], ignore_index=True)
print(f'\nTotaal gecombineerd: {len(df):,} parfums')

#  Kwaliteitsfilter: alleen rijen met gevulde reviews + notes 
# Reviews zijn de belangrijkste signal voor vibe-matching ("for a date" etc).
# Notes blijven nodig omdat ze in de embedding_text gaan.
def _is_nonempty(raw):
    if pd.isna(raw):
        return False
    s = str(raw).strip()
    return s not in ('', '[]', 'nan', 'None')

mask = df['notes'].apply(_is_nonempty)   # alleen notes verplicht; reviews optioneel (meer dekking, #3)

before = len(df)
df = df[mask].reset_index(drop=True)

df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

print(f'\nKwaliteitsfilter (notes verplicht, reviews optioneel): {before:,} → {len(df):,}')
print(f"Rating range na merge: {df['rating'].min():.2f} – {df['rating'].max():.2f}  (mean {df['rating'].mean():.2f})")
df.head(2)

  fragrantica_data/fragnantica_missing_filled.csv: 3,275 rijen
  fragrantica_data/missing_part1_filled.csv: 27,223 rijen
  fragrantica_data/missing_part2_filled.csv: 27,223 rijen
  fragrantica_data/missing_part3_filled.csv: 27,224 rijen
Filled-missing totaal (vóór dedupe): 84,945
Filled-missing na dedupe: 81,670 unieke urls
(Title, designer)-dedupe: 84,144 → 83,755  (-389)

Fragrantica (incl. filled, gededupliceerd): 83,755 parfums
Luckyscent: 3,354 parfums

Totaal gecombineerd: 87,109 parfums

Kwaliteitsfilter (notes verplicht, reviews optioneel): 87,109 → 85,122
Rating range na merge: 0.50 – 5.00  (mean 3.73)


,url,rating,notes,designer,reviews,description,title,gender_raw,style
0,https://www.fragrantica.com/perfume/Franck-Olivier/Baby-Boy-77143.html,2.33,"['Mandarin Orange', 'Freesia', 'Orange Blossom', 'Pear', 'Peony', 'Sandalwood', 'Heliotrope', 'Raspberry']",franck olivier perfumes and colognes,['Smells like a multisex version of Intimately Yours Women by David Beckham take out the sickly sweet almond from th...,Baby Boy by Franck Olivier is a Citrus fragrance for men. This is a new fragrance. Baby Boy was launched in 2022. To...,Baby Boy Franck Olivier for men,NaN,NaN
1,https://www.fragrantica.com/perfume/D-Antol-Parfums/Theatre-Teatr-81737.html,4.33,"['Hawthorn', 'Iris', 'Iris', 'Tuberose', 'Tiare Flower', 'Myrrh', 'Patchouli', 'Styrax', 'Amber']",d’antol’ parfums perfumes and colognes,[],Theatre - Театр by D’Antol’ Parfums is a Amber Floral fragrance for women and men. Theatre - Театр was launched in 2...,Theatre - Театр D’Antol’ Parfums for women and men,NaN,NaN


## 2. Embedding-tekst bouwen

We bouwen per parfum één tekst-blob: **title + brand + gender + (style) + notes + description + user reviews**. Dit is de kerntekst die geembed wordt. Onderweg strippen we scraper-ruis (URLs, "show more", herhaalde leestekens) en gooien we reviews weg die in ≥3 verschillende parfums voorkomen (scraper bleed-through).

In [4]:
import re, html
from collections import defaultdict


# noise stripping 
_URL_RE       = re.compile(r'https?://\S+')
_SHOW_MORE_RE = re.compile(r'\b(?:show|read)\s+more\b', re.I)
_REPEAT_PUNCT = re.compile(r'([!?.,])\1{2,}')
_WHITESPACE   = re.compile(r'\s+')

def clean_review(text):
    """Strip HTML entities, URLs, 'show more' artifacts, repeated punctuation,
    collapse whitespace."""
    if not text:
        return ''
    s = html.unescape(str(text))
    s = _URL_RE.sub(' ', s)
    s = _SHOW_MORE_RE.sub(' ', s)
    s = _REPEAT_PUNCT.sub(r'\1\1', s)  # !!!!! -> !!
    s = _WHITESPACE.sub(' ', s).strip()
    return s


def _raw_review_list(raw):
    """Reviews-kolom -> Python list of raw strings."""
    if pd.isna(raw):
        return []
    s = str(raw).strip()
    if not s:
        return []
    if s.startswith('['):
        try:
            return [str(r) for r in ast.literal_eval(s)]
        except (ValueError, SyntaxError):
            return [s]
    return [s]


#  find reviews shared across multiple perfumes 
MIN_REVIEW_LEN = 50           # drop snippets shorter than this
DUP_THRESHOLD  = 3            # drop reviews that appear in >= N different perfumes

_review_rows = defaultdict(set)
for _i, _raw in df['reviews'].items():
    for _r in _raw_review_list(_raw):
        _cleaned = clean_review(_r).lower()
        if len(_cleaned) >= MIN_REVIEW_LEN:
            _review_rows[_cleaned].add(_i)

DUP_REVIEWS = {k for k, rows in _review_rows.items() if len(rows) >= DUP_THRESHOLD}
print(f'Cross-perfume duplicate review texts (>={DUP_THRESHOLD} perfumes): {len(DUP_REVIEWS):,}')
del _review_rows


# parsers 
def parse_notes(raw):
    """Notes-kolom is óf een string-van-Python-lijst (Fragrantica) óf comma-separated (Luckyscent)."""
    if pd.isna(raw) or not str(raw).strip():
        return ''
    s = str(raw).strip()
    if s.startswith('['):
        try:
            notes = ast.literal_eval(s)
            return ', '.join(str(n) for n in notes)
        except (ValueError, SyntaxError):
            pass
    return s


def parse_reviews(raw, max_reviews=5, max_chars=1500):
    """Clean + filter reviews. Drops: te kort, scraper-noise-only, en reviews
    die in >=DUP_THRESHOLD verschillende parfums voorkomen (scraper bleed-through).
    Sorteert op lengte aflopend zodat we de meest informatieve reviews houden
    binnen het char-budget."""
    items = _raw_review_list(raw)
    if not items:
        return ''

    candidates = []
    for r in items:
        cleaned = clean_review(r)
        if len(cleaned) < MIN_REVIEW_LEN:
            continue
        if cleaned.lower() in DUP_REVIEWS:
            continue
        candidates.append(cleaned)

    # Langste eerst (meeste semantische signaal per char-budget)
    candidates.sort(key=len, reverse=True)

    out, used = [], 0
    for r in candidates[:max_reviews]:
        if used + len(r) > max_chars:
            r = r[:max_chars - used]
        if not r:
            break
        out.append(r)
        used += len(r)
        if used >= max_chars:
            break
    return ' || '.join(out)


def infer_gender(row):
    """Luckyscent heeft schone gender-kolom; Fragrantica zet 'm in de title."""
    raw = row.get('gender_raw')
    if isinstance(raw, str) and raw.strip():
        g = raw.strip().lower()
        if g in ('men', 'women', 'unisex'):
            return g
    title = row.get('title', '')
    if not isinstance(title, str):
        return 'unisex'
    t = title.lower()
    if 'for women and men' in t or 'unisex' in t:
        return 'unisex'
    if 'for women' in t:
        return 'women'
    if 'for men' in t:
        return 'men'
    return 'unisex'


df['notes_text']   = df['notes'].apply(parse_notes)
df['reviews_text'] = df['reviews'].apply(parse_reviews)
df['gender']       = df.apply(infer_gender, axis=1)

# Style is alleen aanwezig voor Luckyscent - voeg toe als het er is
style_part = df.get('style', pd.Series([''] * len(df))).fillna('').astype(str)
style_part = style_part.where(style_part == '', 'Style: ' + style_part + '. ')

# "User reviews:" suffix valt weg als reviews_text leeg is - voorkomt
# dangling phrase die de embedder verwart.
user_reviews_part = df['reviews_text'].where(df['reviews_text'].str.len() == 0,
                                             'User reviews: ' + df['reviews_text'])

# Fragrantica-description is deels een sjabloon (... is a TYPE fragrance for GENDER.
# This is a new fragrance. Top/Middle/Base notes are ...). Dat is ruis + dubbel met
# de notes-kolom, dus strippen we het uit de embedding-tekst (#4).
_BOILER_RE = re.compile(
    r"(This is a new fragrance\.|Top notes are[^.]*\.|Middle notes are[^.]*\.|"
    r"Base notes are[^.]*\.| is an? [^.]*? fragrance for [^.]*?\.)", re.I)
desc_clean = (df['description'].fillna('').astype(str)
              .str.replace(_BOILER_RE, ' ', regex=True)
              .str.replace(r'\s+', ' ', regex=True).str.strip())

df['embedding_text'] = (
    df['title'].fillna('').astype(str) + '. '
    + 'Brand: ' + df['designer'].fillna('').astype(str) + '. '
    + 'Gender: ' + df['gender'] + '. '
    + style_part
    + 'Notes: ' + df['notes_text'] + '. '
    + desc_clean + ' '
    + user_reviews_part
)

df = df.dropna(subset=['embedding_text']).reset_index(drop=True)
df = df[df['embedding_text'].str.len() > 30].reset_index(drop=True)

# Rijen waarvan ALLE reviews gefilterd zijn behouden we - die hebben nog
# steeds signaal uit description + notes. Wel rapporteren hoeveel er zijn.
empty_reviews = (df['reviews_text'].str.len() == 0).sum()
print(f'\nKlaar voor embedding: {len(df):,} parfums')
print(f'Gem. reviews_text lengte: {df["reviews_text"].str.len().mean():.0f} chars')
print(f'Rijen waarvan alle reviews uitgefilterd zijn: {empty_reviews:,} '
      f'({empty_reviews/len(df)*100:.1f}%) - behouden vanwege description+notes')
print('\nVoorbeeld:')
print(df.iloc[0]['embedding_text'][:800])

Cross-perfume duplicate review texts (>=3 perfumes): 1,137

Klaar voor embedding: 85,122 parfums
Gem. reviews_text lengte: 455 chars
Rijen waarvan alle reviews uitgefilterd zijn: 53,066 (62.3%) - behouden vanwege description+notes

Voorbeeld:
Baby Boy Franck Olivier for men. Brand: franck olivier perfumes and colognes. Gender: men. Notes: Mandarin Orange, Freesia, Orange Blossom, Pear, Peony, Sandalwood, Heliotrope, Raspberry. Baby Boy by Franck Olivier Baby Boy was launched in 2022. User reviews: Smells like a multisex version of Intimately Yours Women by David Beckham take out the sickly sweet almond from that and replace with powdery rosy orange. The sandalwood stops it being too sickly and floral. Nice. Not so keen on the bottle as it appears like it is a infants product rather than an adult multisex EDT. Longevity and silage are average. Tiny bottle. Youthful. || Smells like a multisex version of Intimately Yours Women by David Beckham take out the sickly sweet almond from that an

## 3. ChromaDB-collection opbouwen

We embedden elke `embedding_text` met sentence-transformers en slaan ze op in een persistente ChromaDB-collection met cosine-similarity. **Eén vector per parfum** (geen chunking), dus `collection.count()` == aantal parfums. De eerste run berekent alle embeddings (~paar minuten); daarna laden we de opgeslagen collection meteen van schijf.

In [5]:
EMBED_MODEL     = './models/bge-fragrance'   # sterker retrieval-model (was all-MiniLM, #1)
CHROMA_PATH     = './chroma_db'
COLLECTION_NAME = 'fragrances_ft'      # eigen collection per embedding-model (#1/#6)

# Embedding-model laden (eenmalig)
model = SentenceTransformer(EMBED_MODEL)

# Persistente client zodat we embeddings niet elke run opnieuw hoeven te berekenen
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print(f'Bestaande collection geladen: {collection.count():,} parfums')
else:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'},   # cosine similarity
    )
    print(f'Nieuwe (lege) collection aangemaakt: {COLLECTION_NAME}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6333.45it/s]


Bestaande collection geladen: 85,122 parfums


In [6]:
# Vul de collection alleen als-ie nog leeg is (embeddings berekenen duurt even).
# Eén embedding per parfum - geen chunking, dus collection.count() == aantal parfums.
if collection.count() == 0:
    BATCH_SIZE = 500
    total = len(df)
    has_style = 'style' in df.columns

    for start in range(0, total, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total)
        batch = df.iloc[start:end]

        texts = batch['embedding_text'].tolist()
        embeddings = model.encode(texts, batch_size=64, show_progress_bar=False).tolist()

        # ChromaDB-metadata mag geen None bevatten -> missende rating wordt 0.0
        metadatas = []
        for _, row in batch.iterrows():
            meta = {
                'title':  str(row.get('title', '')),
                'brand':  str(row.get('designer', '')),
                'gender': str(row.get('gender', '')),
                'rating': float(row['rating']) if pd.notna(row.get('rating')) else 0.0,
                'notes':  str(row.get('notes_text', '')),
                'url':    str(row.get('url', '')),
            }
            if has_style and pd.notna(row.get('style')):
                meta['style'] = str(row['style'])
            metadatas.append(meta)

        ids = [str(i) for i in batch.index.tolist()]
        collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)

        if (start // BATCH_SIZE) % 10 == 0:
            print(f'  Verwerkt: {end:,} / {total:,}')

    print(f'\nKlaar! {collection.count():,} parfums opgeslagen in ChromaDB.')
else:
    if collection.count() != len(df):
        print(f'LET OP: collection heeft {collection.count():,} items, df {len(df):,}. '
              f'Hernoem COLLECTION_NAME of wis ./chroma_db om te herbouwen.')
    else:
        print(f'Collection al gevuld: {collection.count():,} parfums.')

Collection al gevuld: 85,122 parfums.


## 3b. Split-collecties: notes + reviews

Eén blob per parfum (sectie 3) **middelt het review-signaal weg** en zit bovendien *off-distribution* t.o.v. onze fine-tune (die trainde op **review → notes-passage**). Daarom splitsen we in twee collecties die elk exact één van die twee getrainde vormen zijn:

- **`fragrances_notes`** - 1 doc per parfum: `"title. Brand: …. Notes: …."` (== de fine-tune *positive*). Dekt **alle ~85k**.
- **`fragrances_reviews`** - 1 doc **per recensie** (review-granulariteit), met de parfum-`url` in de metadata zodat we kunnen terugmappen. Alleen de ~38% parfums mét reviews.

`recommender.Recommender(collection, model, reviews_collection=…)` bevraagt beide en fuseert met **coverage-aware RRF + `review_fairness`-multiplier** - zodat de niche-47k zonder reviews niet structureel onder de mainstream zakken. Hetzelfde (fine-tuned) model embedt beide collecties; **geen hertraining nodig**.

> De evaluatie-cellen (5d/5g) meten review-als-query Recall@10. Houd er rekening mee dat een *review-collectie* die metric verhoogt (een vastgehouden recensie matcht zustersrecensies van hetzelfde parfum) - zorg dat de vastgehouden recensie **niet** in de index zit, anders is het triviale leakage.

In [7]:
# === 3b. Split-collecties bouwen: fragrances_notes + fragrances_reviews ===
# Bouwt rechtstreeks uit de in-memory `df` (cellen 1-3). Een standalone variant die uit
# fragrances_ft + de CSV's bouwt staat in build_collections.py (zelfde resultaat).
NOTES_COLL   = 'fragrances_notes'
REVIEWS_COLL = 'fragrances_reviews'
MAX_REVIEWS_PER_PERFUME = 6        # langste-eerst; houdt de reviews-collectie behapbaar
BATCH_SIZE = 500

def notes_passage(row):
    # zelfde vorm als de fine-tune 'positive' (zie finetune_embeddings.py) -> in-distribution
    return f"{row.get('title','')}. Brand: {row.get('designer','')}. Notes: {row.get('notes_text','')}."

def perfume_reviews(row):
    """Losse, gefilterde recensies (len-drempel + cross-perfume dedup), langste-eerst, gecapt."""
    out = []
    for r in _raw_review_list(row.get('reviews')):
        c = clean_review(r)
        if len(c) >= MIN_REVIEW_LEN and c.lower() not in DUP_REVIEWS:
            out.append(c)
    out.sort(key=len, reverse=True)
    return out[:MAX_REVIEWS_PER_PERFUME]

existing = [c.name for c in chroma_client.list_collections()]

# --- fragrances_notes: 1 doc per parfum (alle ~85k) ---
if NOTES_COLL in existing:
    notes_collection = chroma_client.get_collection(NOTES_COLL)
    print(f'{NOTES_COLL}: bestaat al ({notes_collection.count():,} docs)')
else:
    notes_collection = chroma_client.create_collection(NOTES_COLL, metadata={'hnsw:space': 'cosine'})
    for start in range(0, len(df), BATCH_SIZE):
        batch = df.iloc[start:start + BATCH_SIZE]
        texts = [notes_passage(r) for _, r in batch.iterrows()]
        embs  = model.encode(texts, batch_size=64, show_progress_bar=False).tolist()
        ids, metas = [], []
        for i, row in batch.iterrows():
            ids.append(str(i))
            metas.append({
                'title':  str(row.get('title', '')),     'brand': str(row.get('designer', '')),
                'gender': str(row.get('gender', '')),
                'rating': float(row['rating']) if pd.notna(row.get('rating')) else 0.0,
                'notes':  str(row.get('notes_text', '')), 'url': str(row.get('url', '')),
                'n_reviews': len(perfume_reviews(row)),   # voor de coverage-fairness in retrieval
            })
        notes_collection.add(ids=ids, embeddings=embs, documents=texts, metadatas=metas)
        if (start // BATCH_SIZE) % 20 == 0:
            print(f'  notes: {min(start + BATCH_SIZE, len(df)):,}/{len(df):,}')
    print(f'{NOTES_COLL}: klaar ({notes_collection.count():,} docs)')

# --- fragrances_reviews: 1 doc PER recensie (review-granulariteit, url terug naar parfum) ---
if REVIEWS_COLL in existing:
    reviews_collection = chroma_client.get_collection(REVIEWS_COLL)
    print(f'{REVIEWS_COLL}: bestaat al ({reviews_collection.count():,} docs)')
else:
    reviews_collection = chroma_client.create_collection(REVIEWS_COLL, metadata={'hnsw:space': 'cosine'})
    bids, btxt, bmeta = [], [], []

    def _flush():
        if not btxt:
            return
        embs = model.encode(btxt, batch_size=64, show_progress_bar=False).tolist()
        reviews_collection.add(ids=bids, embeddings=embs, documents=btxt, metadatas=bmeta)
        bids.clear(); btxt.clear(); bmeta.clear()

    for i, row in df.iterrows():
        for j, rv in enumerate(perfume_reviews(row)):
            bids.append(f'{i}-{j}')
            btxt.append(rv)                       # recensie = document (geen query-prefix)
            bmeta.append({'url': str(row.get('url', '')),    'gender': str(row.get('gender', '')),
                          'title': str(row.get('title', '')), 'brand': str(row.get('designer', ''))})
            if len(btxt) >= 1000:
                _flush()
    _flush()
    print(f'{REVIEWS_COLL}: klaar ({reviews_collection.count():,} review-docs)')

  notes: 500/85,122
  notes: 10,500/85,122
  notes: 20,500/85,122
  notes: 30,500/85,122
  notes: 40,500/85,122
  notes: 50,500/85,122
  notes: 60,500/85,122
  notes: 70,500/85,122
  notes: 80,500/85,122
fragrances_notes: klaar (85,122 docs)
fragrances_reviews: klaar (130,274 review-docs)


## 4. Hybride retrieval (vector + BM25)

Vector search vindt parfums op **vibe** ("spicy for a date"), maar als de gebruiker een **exacte note of merknaam** noemt ("oud", "Dior") wil je die hard matchen - en daar is een embedding soms te "wollig" voor. Daarom draaien we twee retrievers naast elkaar:

1. **ChromaDB** (semantisch) - vibe-match op de volledige `embedding_text`.
2. **BM25** (lexicaal) - keyword-match op `notes + title + brand`. Een query met "oud" tilt elk parfum dat letterlijk oud bevat naar boven.

We fuseren beide ranglijsten met **Reciprocal Rank Fusion** (RRF): `score(d) = Σ 1/(k + rank_d)`. RRF werkt op posities, niet op scores, dus we hoeven de twee totaal verschillende score-schalen niet te normaliseren. Daarna nog de lichte rating-rerank en dedup op `url`.

> Tip: de query expansion zet vibe-woorden om in note-termen ("spicy" → "cinnamon pepper saffron..."), waardoor óók BM25 mee profiteert.

In [8]:
# === Retrieval-core (gedeeld met app.py via recommender.py) ===
# Een bron van waarheid: deze notebook (lab) en de webapp gebruiken exact dezelfde
# Recommender. Catalogus + BM25 worden uit de ChromaDB-collectie geladen.
from recommender import Recommender, expand_query, tokenize

rec = Recommender(collection, model)
recommend = rec.recommend          # zelfde call: recommend('query', top_k=..., gender=..., use_bm25=...)
print(f'Recommender klaar - {len(rec.catalog):,} parfums (ChromaDB + BM25, gedeeld met app.py).')

Recommender klaar - 85,122 parfums (ChromaDB + BM25, gedeeld met app.py).


In [9]:
recommend('spicy perfume for a date', top_k=5, min_rating=3.5)

,title,brand,gender,rating,notes,url,score
0,X Twist Saffron Clive Christian for women and men,clive christian perfumes and colognes,unisex,4.37,"Pink Pepper, Ginger, Saffron, Cinnamon, Orris, Sandalwood, Musk",https://www.fragrantica.com/perfume/Clive-Christian/X-Twist-Saffron-39249.html,0.625
1,Warm Black Zara for men,zara perfumes and colognes,men,4.13,"Ginger, Vanilla, Tonka Bean",https://www.fragrantica.com/perfume/Zara/Warm-Black-62359.html,0.571
2,Tobacco Collection Rich Warm Addictive 2018 Zara for men,zara perfumes and colognes,men,4.55,"Rum, Peony, Bourbon Vanilla",https://www.fragrantica.com/perfume/Zara/Tobacco-Collection-Rich-Warm-Addictive-2018-52183.html,0.454
3,Saffron Rouge Lvnea for women and men,lvnea perfumes and colognes,unisex,3.50,"Saffron, Nard Himalayan (Jatamansi), Calamus, Rose, Cinnamon",https://www.fragrantica.com/perfume/Lvnea/Saffron-Rouge-52470.html,0.446
4,Zaffran PK Perfumes for women and men,pk perfumes perfumes and colognes,unisex,3.60,"Saffron, Costus, Cloves, Tobacco, Vetiver, Cinnamon, Tangerine, Patchouli, Mandarin Orange, Sandalwood, Blood Orange...",https://www.fragrantica.com/perfume/PK-Perfumes/Zaffran-16541.html,0.445


In [10]:
recommend('fresh summer office scent, not too strong', top_k=5, gender='men', min_rating=3.5)

,title,brand,gender,rating,notes,url,score
0,Eau Fraîche Citrus Marino Angel Schlesser for men,angel schlesser perfumes and colognes,men,3.56,"Lemon, Sea Notes, Bergamot, Tangerine, Galbanum, Red Thyme, Black Pepper, Vetiver, Lavender, Geranium, Patchouli, Mu...",https://www.fragrantica.com/perfume/Angel-Schlesser/Eau-Fraiche-Citrus-Marino-60377.html,0.566
1,Cool Water Happy Summer Man Davidoff for men,davidoff perfumes and colognes,men,3.55,"Grapefruit, Lemon, Orange, Mandarin Orange, Watermelon, Passionfruit, Pineapple, Green Apple, Jasmine, Violet Leaf, ...",https://www.fragrantica.com/perfume/Davidoff/Cool-Water-Happy-Summer-Man-44590.html,0.547
2,Tommy Citrus Brights Tommy Hilfiger for men,tommy hilfiger perfumes and colognes,men,3.55,"Lemon, Lime, Bergamot, Pine Tree Needles, Orange, Lavender, Ginger, Mint, Haitian Vetiver, Incense, Pink Pepper",https://www.fragrantica.com/perfume/Tommy-Hilfiger/Tommy-Citrus-Brights-34947.html,0.544
3,Poseidon Fresh Line for men,fresh line perfumes and colognes,men,3.55,"Cedar, Rosemary, Spicy Mint, Bergamot, Lemon, Fennel, Lavender, Ginger",https://www.fragrantica.com/perfume/Fresh-Line/Poseidon-32752.html,0.525
4,Uzon Fresh Jequiti for men,jequiti perfumes and colognes,men,3.75,"Bergamot, Green Leaves, Lavender, Floral Fruity Notes, Amber, Black Pepper",https://www.fragrantica.com/perfume/Jequiti/Uzon-Fresh-25244.html,0.489


In [11]:
recommend('elegant floral for a wedding', top_k=5, gender='women', min_rating=4.0)

,title,brand,gender,rating,notes,url,score
0,Aroma Flower Harmony Marbert for women,marbert perfumes and colognes,women,4.00,"Jasmine, Iris, Rose, Lily-of-the-Valley, Vanilla",https://www.fragrantica.com/perfume/Marbert/Aroma-Flower-Harmony-29996.html,0.646
1,Essenses 45 Nuancielo for women,nuancielo perfumes and colognes,women,4.00,"Peony, Litchi, Freesia, Rose, Lily-of-the-Valley, Magnolia, Virginia Cedar, Amber",https://www.fragrantica.com/perfume/Nuancielo/Essenses-45-56728.html,0.492
2,Iris Absolute Washington Tremlett for women,washington tremlett perfumes and colognes,women,4.37,"Bulgarian Rose, Jasmine, Iris, Heliotrope, Lily-of-the-Valley, White Musk, Tonka Bean, Vanilla",https://www.fragrantica.com/perfume/Washington-Tremlett/Iris-Absolute-6983.html,0.435
3,Under the Elegant Coconut Palms Alkemia Perfumes for women,alkemia perfumes perfumes and colognes,women,4.25,"Coconut, Coconut Nectar, Vanilla Pod, Ginger, Spicy Notes, Musk, Banana Leaf, Tobacco Blossom, White Amber, Frangipani",https://www.fragrantica.com/perfume/Alkemia-Perfumes/Under-the-Elegant-Coconut-Palms-54273.html,0.431
4,Floralia Iris Imperiale Borsari for women,borsari perfumes and colognes,women,4.50,"Black Currant, Bergamot, Grapefruit, Peach, Jasmine, Magnolia, Peony, Lemongrass, Musk, Sandalwood, Vanilla",https://www.fragrantica.com/perfume/Borsari/Floralia-Iris-Imperiale-64233.html,0.386


## 5. Evaluatie

We meten **Precision@5**: van de top-5 aanbevelingen, hoeveel bevatten minstens één van de verwachte notes? Zelfde proxy als in `fragrantica_chatbot.ipynb`. We zetten **hybrid naast vector-only** (`use_bm25=False`) om te zien of de BM25-pass echt helpt - vooral op de exacte-note query "oud".

> Kanttekening: note-overlap is een welwillende maat voor een keyword-retriever en een zwakke proxy voor echte vibe-queries ("for a date"). Zie het als sanity check, niet als eindoordeel - voor vibe-matching telt uiteindelijk de subjectieve beoordeling.

In [12]:
# === Precision@5 op note-relevantie - hybrid vs vector-only ===
# Ground truth = lexicale note-overlap (zelfde proxy als fragrantica_chatbot.ipynb).
test_cases = [
    {'query': 'fresh citrus summer beach',     'expected_notes': ['citrus', 'lemon', 'bergamot', 'lime', 'orange', 'grapefruit', 'neroli']},
    {'query': 'dark woody oud oriental',        'expected_notes': ['oud', 'agarwood', 'cedar', 'sandalwood', 'patchouli', 'vetiver']},
    {'query': 'sweet vanilla warm gourmand',    'expected_notes': ['vanilla', 'caramel', 'amber', 'tonka', 'honey']},
    {'query': 'floral rose jasmine feminine',   'expected_notes': ['rose', 'jasmine', 'lily', 'iris', 'peony', 'violet']},
    {'query': 'spicy pepper saffron',           'expected_notes': ['pepper', 'saffron', 'cinnamon', 'cardamom', 'clove', 'ginger']},
    {'query': 'oud',                            'expected_notes': ['oud', 'agarwood']},   # exacte-note test: hier moet BM25 het verschil maken
]


def precision_at_k(query, expected_notes, k=5, use_bm25=True):
    res = recommend(query, top_k=k, use_bm25=use_bm25)
    if res.empty:
        return 0.0
    hits = sum(any(n in str(notes).lower() for n in expected_notes) for notes in res['notes'])
    return hits / len(res)


eval_rows = []
for case in test_cases:
    p_vec    = precision_at_k(case['query'], case['expected_notes'], use_bm25=False)
    p_hybrid = precision_at_k(case['query'], case['expected_notes'], use_bm25=True)
    eval_rows.append({
        'query':       case['query'],
        'P@5 vector':  round(p_vec, 2),
        'P@5 hybrid':  round(p_hybrid, 2),
    })

eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))
print(f"\nGemiddelde P@5 - vector-only: {eval_df['P@5 vector'].mean():.2f}  |  hybrid: {eval_df['P@5 hybrid'].mean():.2f}")

                       query  P@5 vector  P@5 hybrid
   fresh citrus summer beach         1.0         1.0
     dark woody oud oriental         1.0         1.0
 sweet vanilla warm gourmand         1.0         1.0
floral rose jasmine feminine         1.0         1.0
        spicy pepper saffron         1.0         1.0
                         oud         1.0         1.0

Gemiddelde P@5 - vector-only: 1.00  |  hybrid: 1.00


### Wat zegt deze evaluatie?

**Kolommen** = twee versies van de zoeker:
- `P@5 vector` - alleen ChromaDB (semantisch)
- `P@5 hybrid` - ChromaDB + BM25 (keyword)

**P@5** = van de top-5 aanbevelingen, hoeveel bevatten minstens één van de verwachte notes? `1.0` = 5/5, `0.8` = 4/5, `0.6` = 3/5.

**Per query:**

| query | vector | hybrid | wat gebeurde er |
|---|---|---|---|
| fresh citrus summer beach | 1.0 | 1.0 | beide perfect (5/5) |
| dark woody oud oriental | 1.0 | 1.0 | beide perfect |
| **sweet vanilla warm gourmand** | 0.6 | 1.0 | vector 3/5, **hybrid 5/5** → BM25 voegde 2 toe |
| floral rose jasmine feminine | 1.0 | 1.0 | beide perfect |
| spicy pepper saffron | 1.0 | 1.0 | beide perfect |
| **oud** | 0.8 | 1.0 | vector 4/5, **hybrid 5/5** → BM25 voegde 1 toe |

**Conclusie:** hybrid is overal gelijk aan of beter dan vector-only (gemiddeld 0.90 → 1.00). Het verbeterde precies de twee zwakkere gevallen (`sweet vanilla` en `oud`) en maakte niets slechter. De BM25-pass doet dus wat-ie moet doen: exacte note-queries zoals "oud" hard matchen.

**Kanttekening - niet te enthousiast worden:** deze metriek meet alleen of een resultaat een note-*woord* uit de query bevat, en BM25 zoekt nu juist op woorden - dus de metriek speelt BM25 in de kaart (deels circulair). Ze zegt "hybrid ≥ vector", niet "de aanbevelingen zijn goed". Met maar 6 note-achtige queries en geen echte vibe-queries ("voor een date") is dit een sanity check, geen hard bewijs. Voor een betrouwbaar oordeel: voeg met de hand beoordeelde vibe-queries toe en gebruik een strengere metriek (bijv. fractie gedekte notes of nDCG).

## 5b. Strengere metrieken (coverage, nDCG)

P@5 is grof (>=1 verwachte note volstaat). Hier zetten we **vector-only naast hybrid** met twee strengere maten:

- **coverage@5** - welk deel van alle verwachte notes de top-5 samen dekt (beloont variatie).
- **nDCG@5** - rang-bewust en graded (relevance = aantal verwachte notes per hit); beloont de beste matches bovenaan.

In [13]:
# === 5b. Strengere metrieken: vector vs hybrid ===
import math
import numpy as np

def _matched(notes, exp):
    s = str(notes).lower()
    return [n for n in exp if n in s]

def p_at_k(res, exp):
    return 0.0 if res.empty else sum(bool(_matched(n, exp)) for n in res['notes']) / len(res)

def coverage_at_k(res, exp):
    found = set()
    for n in res['notes']:
        found.update(_matched(n, exp))
    return len(found) / len(exp)

def ndcg_at_k(res, exp):
    rels = [len(_matched(n, exp)) for n in res['notes']]
    dcg  = sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rels, reverse=True)))
    return dcg / idcg if idcg else 0.0

summary = []
for name, ub in [('vector', False), ('hybrid', True)]:
    ps, cov, nd = [], [], []
    for case in test_cases:
        res = recommend(case['query'], top_k=5, use_bm25=ub)
        ps.append(p_at_k(res, case['expected_notes']))
        cov.append(coverage_at_k(res, case['expected_notes']))
        nd.append(ndcg_at_k(res, case['expected_notes']))
    summary.append({'variant': name, 'P@5': round(np.mean(ps), 3),
                    'coverage@5': round(np.mean(cov), 3), 'nDCG@5': round(np.mean(nd), 3)})

print(pd.DataFrame(summary).to_string(index=False))

variant  P@5  coverage@5  nDCG@5
 vector  1.0       0.873   0.952
 hybrid  1.0       0.919   0.922


## 5c. Vibe-queries (handmatig beoordelen)

Note-overlap zegt niks voor échte vibe-queries ("iets om mee te lezen op een regenachtige dag"). Daar is geen note-ground-truth voor - die moet je zelf beoordelen. Run de cel, bekijk de top-5 per query, vul daarna `judgments` met vijf 0/1-oordelen (past / past niet) en run opnieuw. De cel rekent dan je eigen P@5 uit.

In [14]:
# === 5c. Vibe-queries: top-5 tonen + handmatige P@5 ===
# Voor deze queries bestaat geen note-ground-truth. Run de cel, bekijk de top-5,
# vul daarna 'judgments' met vijf 0/1-oordelen per query en run opnieuw.
vibe_queries = [
    'something cozy to wear while reading on a rainy day',
    'confident scent for a job interview',
    'romantic perfume for a winter date night',
    'a clean everyday signature scent',
]

for q in vibe_queries:
    print('=' * 72)
    print('QUERY:', q)
    res = recommend(q, top_k=5)
    for n, r in enumerate(res.itertuples()):
        print(f'  [{n}] {r.title}  ({r.rating}/5)')
        print(f'      {str(r.notes)[:75]}')

# --- Vul per query vijf 0/1-oordelen in (volgorde = [0..4] hierboven) ---
judgments = {
    'something cozy to wear while reading on a rainy day': [],   # bv. [1, 0, 1, 1, 0]
    'confident scent for a job interview':                 [],
    'romantic perfume for a winter date night':            [],
    'a clean everyday signature scent':                    [],
}

scored = {q: sum(v) / len(v) for q, v in judgments.items() if v}
print('\n' + '=' * 72)
if scored:
    for q, p in scored.items():
        print(f'P@5 = {p:.2f}  | {q}')
    print(f'\nGemiddelde P@5 (vibe, handmatig): {sum(scored.values()) / len(scored):.2f}')
else:
    print('Nog geen oordelen ingevuld - vul de judgments-lijsten met 0/1 en run opnieuw.')

QUERY: something cozy to wear while reading on a rainy day
  [0] Colorful Scent Rainy Etude House for women  (0.0/5)
      Grass, Cypress, Rain Notes, White Orchid, Musk, Amber, Moss
  [1] Rainy City M.INT for women and men  (0.0/5)
      Green Tea, Lemon, Bergamot, Black Tea, Cedar, Guaiac Wood, Vetiver, Musk, W
  [2] Rainy Forest Дождливый Лес OsmoGenes Perfumes for women and men  (0.0/5)
      Rain Notes, Fern, Mushroom, Fir Resin, Mushroom, Green Leaves, Oakmoss, Fir
  [3] In The Rain Floraïku for women and men  (0.0/5)
      Bergamot, Cedar, Woody Notes, Musk
  [4] Cozy On A Snow Day Find Your Happy Place for women  (0.0/5)
      Cashmere Musk, Vanilla, Sandalwood, Whipped Cream
QUERY: confident scent for a job interview
  [0] Quantium Confident Avon for women and men  (0.0/5)
      Mandarin Orange, Black Pepper, Woody Notes
  [1] Confident My Geisha for women and men  (0.0/5)
      Saffron, Nutmeg, Tropical Fruit, Bergamot, Lavender, Magnolia, Oud, Jasmine
  [2] Confidence Rasasi

## 5d. Recall@k + MRR (recensie-als-query)

Zelf-gesuperviseerd, zonder handmatige labels. Voor een steekproef parfums nemen we een **recensie die niet in `embedding_text` zit** (vermijdt leakage) en gebruiken die als query. Komt de juiste parfum in de top-k terug? Dit meet direct of de taal van echte meningen de bedoelde parfum vindt.

- **Recall@k** - fractie waar de juiste parfum in de top-k zit.
- **MRR@k** - gemiddelde van 1/rang (beloont hoog plaatsen).

In [15]:
# === 5d. Recall@k + MRR - recensie-als-query (leave-reviews-out) ===
import numpy as np

K, N = 10, 300
sample = df.sample(min(N, len(df)), random_state=0)

hits, rr, used = [], [], 0
for _, row in sample.iterrows():
    indexed = str(row.get('reviews_text', ''))
    held = [clean_review(r) for r in _raw_review_list(row.get('reviews'))]
    held = [h for h in held if len(h) >= 40 and h not in indexed]   # niet geindexeerd -> geen leakage
    if not held:
        continue
    res = recommend(held[0], top_k=K, expand=False, rating_weight=0.0)  # pure retrieval op de recensietekst
    urls = list(res['url']) if not res.empty else []
    if row['url'] in urls:
        rank = urls.index(row['url']) + 1
        hits.append(1); rr.append(1.0 / rank)
    else:
        hits.append(0); rr.append(0.0)
    used += 1

print(f'Recensie-als-query op {used} parfums (sample {len(sample)}):')
print(f'  Recall@{K}: {np.mean(hits):.3f}   (juiste parfum in top-{K})')
print(f'  MRR@{K}:    {np.mean(rr):.3f}   (1/rang; hoger = beter geplaatst)')

KeyboardInterrupt: 

## 5e. Personalisatie-eval: leave-one-out (merk-proxy)

Echte personalisatie meet je met user-logs (die we niet hebben). **Proxy**: een pseudo-user houdt van een merk -> bouw een profiel uit 3 parfums van dat merk, hou de 4e achter, en kijk of *profiel-only* retrieval die terugvindt. Vergelijk met een random baseline. Personalisatie voegt signaal toe als WITH duidelijk > baseline. (Zwakke proxy, maar draait op onze data; #2.)

In [ ]:
# === 5e. Personalisatie leave-one-out (merk-proxy) ===
import numpy as np

rng = np.random.default_rng(0)
groups = {b: list(idx) for b, idx in df.groupby('designer').groups.items()
          if isinstance(b, str) and len(idx) >= 4}
brands = list(groups); rng.shuffle(brands); brands = brands[:150]
all_urls = df['url'].tolist()

K = 20
with_hits, base_hits, n = [], [], 0
for b in brands:
    pick = list(rng.choice(groups[b], size=4, replace=False))
    seeds, held = pick[:3], pick[3]
    held_url = df.iloc[held]['url']
    profile = [{'id': str(s), 'rating': 5.0} for s in seeds]

    res = recommend('', profile=profile, top_k=K, rating_weight=0.0)   # profiel-only
    with_hits.append(int(held_url in set(res['url'])) if not res.empty else 0)
    base_hits.append(int(held_url in set(rng.choice(all_urls, size=K, replace=False))))
    n += 1

print(f'Leave-one-out (merk-proxy), {n} pseudo-users, K={K}:')
print(f'  Recall@{K} met profiel : {np.mean(with_hits):.3f}')
print(f'  Recall@{K} baseline    : {np.mean(base_hits):.4f}  (random)')
print('  -> profiel werkt als bovenste duidelijk hoger is dan de baseline.')

## 5f. Knoppen tunen (grid op review-als-query Recall@10)

Klein grid over `rrf_k` en `rating_weight`, gescoord op de leakage-vrije review-als-query Recall@10 (#5). Geeft een startpunt i.p.v. handmatig gekozen defaults.

In [ ]:
# === 5f. Mini grid-search ===
import numpy as np

K = 10
sample = df.sample(min(120, len(df)), random_state=1)
pairs = []
for _, row in sample.iterrows():
    indexed = str(row.get('reviews_text', ''))
    held = [clean_review(r) for r in _raw_review_list(row.get('reviews'))]
    held = [h for h in held if len(h) >= 40 and h not in indexed]
    if held:
        pairs.append((held[0], row['url']))

def recall(rrf_k, rating_weight):
    h = []
    for q, target in pairs:
        res = recommend(q, top_k=K, expand=False, rrf_k=rrf_k, rating_weight=rating_weight)
        h.append(int(target in set(res['url'])) if not res.empty else 0)
    return float(np.mean(h))

print(f'Grid op {len(pairs)} paren (Recall@{K}):')
for rk in (30, 60, 100):
    for rw in (0.0, 0.15):
        print(f'  rrf_k={rk:3d}  rating_weight={rw:.2f}  ->  Recall@{K} = {recall(rk, rw):.3f}')

## 5g. bge-base vs fine-tuned (naast elkaar)

Zelfde review-als-query test (Recall@10 / MRR@10) op beide modellen/collecties tegelijk, zodat je ziet of fine-tunen écht wint. Vereist dat **beide** collecties bestaan: `fragrances_bge` (basis) en `fragrances_ft` (fine-tuned). Bestaat er één niet, dan toont die rij een melding (bouw 'm eerst).

In [ ]:
# === 5g. bge-base vs fine-tuned ===
from recommender import Recommender
import numpy as np

K, N = 10, 600
_sample = df.sample(min(N, len(df)), random_state=0)
pairs = []
for _, row in _sample.iterrows():
    indexed = str(row.get('reviews_text', ''))
    held = [clean_review(r) for r in _raw_review_list(row.get('reviews'))]
    held = [h for h in held if len(h) >= 40 and h not in indexed]
    if held:
        pairs.append((held[0], row['url']))

def _eval(rec_obj):
    hits, rr = [], []
    for q, target in pairs:
        res = rec_obj.recommend(q, top_k=K, expand=False, rating_weight=0.0)
        urls = list(res['url']) if not res.empty else []
        if target in urls:
            r = urls.index(target) + 1
            hits.append(1); rr.append(1.0 / r)
        else:
            hits.append(0); rr.append(0.0)
    return np.mean(hits), np.mean(rr)

CANDIDATES = [
    ('bge-base',   'BAAI/bge-small-en-v1.5', 'fragrances_bge'),
    ('fine-tuned', './models/bge-fragrance', 'fragrances_ft'),
]

rows = []
for name, model_path, coll_name in CANDIDATES:
    try:
        coll = chroma_client.get_collection(coll_name)
        r = Recommender(coll, SentenceTransformer(model_path))
        rec10, mrr = _eval(r)
        rows.append({'model': name, 'collection': coll_name,
                     'Recall@10': round(rec10, 3), 'MRR@10': round(mrr, 3)})
    except Exception as e:
        rows.append({'model': name, 'collection': coll_name,
                     'Recall@10': f'(mist: {type(e).__name__})', 'MRR@10': '-'})

print(f'Vergelijking op {len(pairs)} review-paren:')
print(pd.DataFrame(rows).to_string(index=False))

## 6. LLM-laag (lokaal via Ollama)

De retrieval levert de top-k parfums; een lokaal taalmodel giet dat in een vloeiende aanbeveling. We draaien **Qwen2.5 7B** via **Ollama** - lokaal, gratis, geen API-key, en het gebruikt automatisch de GPU (RTX 5050).

**Eenmalige setup (buiten de notebook):**
1. Installeer Ollama: https://ollama.com/download (Windows-installer - signed, dus geen app-control-gedoe).
2. Haal het model op: `ollama pull qwen2.5:7b` (~4.7 GB). Ollama draait daarna als service op `localhost:11434`.

Ollama spreekt het **OpenAI-API-formaat**, dus we gebruiken de `openai`-client met een andere `base_url` - exact het patroon van de OpenAI-optie in `fragrantica_chatbot.ipynb`, maar lokaal. Draait Ollama niet, dan valt `genereer_aanbeveling()` netjes terug op een kale lijst.

> Het model **herformuleert alleen** de opgehaalde parfums (grounding via de prompt), het verzint geen parfums of notes. Zo blijft de aanbeveling trouw aan de hybride retrieval.

In [ ]:
# === 6. LLM-laag: lokale tekstgeneratie via Ollama (Qwen2.5 7B) ===
# Ollama spreekt het OpenAI-API-formaat, dus we hergebruiken de openai-client
# met een andere base_url. Geen echte API-key nodig (de waarde wordt genegeerd).
from openai import OpenAI

OLLAMA_URL   = 'http://localhost:11434/v1'
OLLAMA_MODEL = 'qwen2.5:7b'
llm = OpenAI(base_url=OLLAMA_URL, api_key='ollama')


def _format_kandidaten(results: pd.DataFrame) -> str:
    regels = []
    for n, r in enumerate(results.itertuples(), 1):
        rating = f'{r.rating:.1f}/5' if r.rating else 'geen rating'
        regels.append(f'{n}. {r.title} | merk: {r.brand} | {r.gender} | {rating}\n'
                      f'   Notes: {r.notes}')
    return '\n'.join(regels)


def genereer_aanbeveling(query: str, results: pd.DataFrame | None = None,
                         top_k: int = 5, gender: str | None = None,
                         min_rating: float = 0.0, language: str = 'English',
                         model: str = OLLAMA_MODEL) -> str:
    """Haalt (indien nodig) de top-k parfums op en laat het lokale LLM er een
    vloeiende aanbeveling van schrijven. Valt netjes terug op een kale lijst
    als Ollama niet draait. Het model herformuleert alleen de opgehaalde
    parfums (grounding) - het verzint niks."""
    if results is None:
        results = recommend(query, top_k=top_k, gender=gender, min_rating=min_rating)
    if results.empty:
        return 'Geen passende parfums gevonden voor deze beschrijving.'

    kandidaten = _format_kandidaten(results)
    system = (
        'You are an expert, friendly fragrance advisor helping someone choose a perfume. '
        'Work ONLY from the candidate perfumes provided - never invent perfumes, brands or notes, '
        "and only mention notes that actually appear in a candidate's note list. "
        f'Discuss ALL {len(results)} candidates (do not drop any); they are already ranked by '
        'relevance, so lead with the strongest. '
        "For each pick, connect it to the user's request (occasion, vibe, season) through its notes - "
        'explain WHY it fits instead of just restating the notes. '
        'Be honest: if a match is weak, say so briefly rather than overselling. '
        'Mention the rating only when it is a clear plus (about 4/5 or higher), and respect any requested gender. '
        'Avoid generic marketing fluff; be specific and concrete. '
        f'Answer in {language}: open with one short sentence tying back to the request, then a numbered '
        'list - each item the perfume name in bold followed by one or two sentences.'
    )
    user = f'User request: "{query}"\n\nCandidate perfumes:\n{kandidaten}'

    try:
        resp = llm.chat.completions.create(
            model=model,
            messages=[{'role': 'system', 'content': system},
                      {'role': 'user', 'content': user}],
            temperature=0.7,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        kale_lijst = '\n'.join(f'- {r.title} ({r.brand}) - {r.notes}'
                               for r in results.itertuples())
        return (f'[LLM niet bereikbaar: {type(e).__name__}. Draait Ollama? '
                f'Installeer via https://ollama.com, dan `ollama pull {model}`.]\n\n'
                f'Top {len(results)} voor "{query}":\n{kale_lijst}')


# Demo (vereist een draaiende Ollama + `ollama pull qwen2.5:7b`)
print(genereer_aanbeveling('I am a gay bottom and Im going to a gay club and Id like to pick up a guy for a hookup', top_k=5, min_rating=3.5))

## 7. Webapp (FastAPI + frontend)

De interactieve demo draait **buiten de notebook**, als losse webapp op dezelfde ChromaDB-collectie:

- `app.py` - FastAPI-backend: smaakprofiel (gewogen embedding van favoriete parfums) + gesprek + aanbeveling via lokale Ollama (`qwen2.5:7b`).
- `static/index.html` - frontend (vanilla JS, geen build-step).

**Run:**
```bash
pip install fastapi uvicorn
# Ollama moet draaien:  ollama run qwen2.5:7b
python app.py          # open http://127.0.0.1:8000
```

De app laadt embeddings + metadata rechtstreeks uit `./chroma_db` - **geen CSV-herbouw nodig**. Run dus eerst secties 1-3 een keer zodat de collectie op schijf bestaat.

## Volgende stappen

- **Embeddings fine-tunen** (`finetune_embeddings.py`): trainen op (review -> parfum) paren voor fragrance-aware vectoren - grootste kwaliteitswinst, vereist GPU/CUDA-torch. Daarna `EMBED_MODEL` op het lokale pad zetten en herbouwen.
- **Deploy**: de webapp draait lokaal met lokale Ollama. Online hosten = LLM vervangen door een gehoste API, of Ollama meeleveren.
- **Echte personalisatie-eval**: 5e is een merk-proxy; met user-logs wordt het een betrouwbare leave-one-out.
- **Note-niveau feedback**: dislikes werken nu op parfum-niveau (hele blob); per note zou fijner zijn.